# PRISM - mechanism analysis across both transfer designs

The Camelyon17 run established that raw target ECE rises with the source label
fraction in 28 of 32 label-definition-shift combinations but only 10 of 160
covariate-shift ones. Two designs, opposite directions.

That contrast is real but it is not yet an explanation, and a reviewer will
notice that the two designs differ in more than the kind of shift: the
label-definition pairs come from MHIST, PCam, CRC and BRACS and transfer at
AUROC near 0.5, while the hospital pairs come from Camelyon17 and transfer at
0.88 to 0.996. The kind of shift and the difficulty of the transfer are
confounded across designs.

**The confound is separable inside Camelyon17.** There the dataset, the task
and the kind of shift are all held constant, and only transfer quality varies
between models and pairs. CLIP has the worst OOD AUROC at 0.88 and shows the
effect in 8 of 20 pairs; the models transferring at 0.99 show it in 0 or 1 of
20. If that relationship holds across all 192 combinations from both designs,
the mechanism is transfer failure rather than the kind of shift, and the
label-definition pairs are simply the most extreme way to produce it.

This notebook tests that. It is the analysis that decides how the paper states
its central claim:

| outcome | what the paper claims |
|---|---|
| ECE trend tracks transfer AUROC across both designs | reverse scaling is a signature of failing transfer; label-definition shift is one route to it |
| it tracks the design and not transfer quality | the two shift types differ intrinsically, and the mechanism is left open |
| it tracks degeneracy above all | the effect is an artefact of single-class collapse and must be stated as such |

All three are reportable. CPU only, a few minutes, no re-runs.

In [1]:
import os, warnings
import numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr, mannwhitneyu
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/PRISM/results_v2'
FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]

lab = pd.read_csv(f'{OUT_DIR}/ood_all_v2.csv')          # label-definition shift
cov = pd.read_csv(f'{OUT_DIR}/camelyon17_transfer.csv') # covariate shift

print(f'label-definition rows {len(lab):,}, pairs {lab["pair"].nunique()}')
print(f'covariate rows        {len(cov):,}, pairs {cov["pair"].nunique()}')
print(f'\ncolumns shared: '
      f'{sorted(set(lab.columns) & set(cov.columns))}')

Mounted at /content/drive
label-definition rows 576, pairs 4
covariate rows        2,880, pairs 20

columns shared: ['auroc', 'brier', 'degeneracy_share', 'degenerate', 'ece_adaptive', 'ece_fixed', 'ece_scaled_adaptive', 'ece_scaled_fixed', 'f1_macro', 'fraction', 'model', 'n_train', 'pair', 'seed', 'src', 'temperature_src', 'tgt']


## 1. One table, 192 combinations

For every (model, pair) in either design, summarise the ECE trend across label
fractions and the properties that might explain it.

In [2]:
def summarise(df, design):
    rows = []
    for m in df['model'].unique():
        for p in df['pair'].unique():
            d = df[(df.model == m) & (df.pair == p)]
            if d.empty:
                continue
            e = d.groupby('fraction')['ece_fixed'].mean().sort_index()
            a = d.groupby('fraction')['auroc'].mean().sort_index()
            if len(e) < 2:
                continue
            # slope of ECE against log label fraction, a scale-free trend
            x = np.log10(e.index.values)
            slope = float(np.polyfit(x, e.values, 1)[0])
            rows.append(dict(
                design=design, model=m, pair=p,
                ece_lo=e.iloc[0], ece_hi=e.iloc[-1],
                ece_delta=e.iloc[-1] - e.iloc[0],
                ece_slope=slope,
                rises=bool(e.iloc[-1] > e.iloc[0]),
                auroc_lo=a.iloc[0], auroc_hi=a.iloc[-1],
                auroc_mean=float(a.mean()),
                auroc_dist=float(abs(a.mean() - 0.5)),
                degeneracy=float(d['degeneracy_share'].mean()),
                degenerate_share=float(d['degenerate'].mean())))
    return pd.DataFrame(rows)


S = pd.concat([summarise(lab, 'label-definition'),
               summarise(cov, 'covariate')], ignore_index=True)
S.to_csv(f'{OUT_DIR}/mechanism_summary.csv', index=False)

print(f'{len(S)} (model, pair) combinations\n')
print(S.groupby('design').agg(
    n=('rises','size'), rises=('rises','sum'),
    mean_delta=('ece_delta','mean'), mean_slope=('ece_slope','mean'),
    mean_auroc=('auroc_mean','mean'),
    mean_degeneracy=('degenerate_share','mean')).round(4).to_string())

192 (model, pair) combinations

                    n  rises  mean_delta  mean_slope  mean_auroc  mean_degeneracy
design                                                                           
covariate         160     10     -0.1492     -0.0743      0.9700           0.0069
label-definition   32     28      0.0820      0.0405      0.5382           0.5729


## 2. Does the ECE trend track transfer quality?

The central question. If reverse scaling is a signature of failing transfer,
then the ECE slope should be negatively related to transfer AUROC: good
transfer, calibration improves with labels; poor transfer, it worsens.

In [3]:
print('=== across all 192 combinations ===\n')
for xcol, lab_ in [('auroc_mean', 'transfer AUROC'),
                   ('degenerate_share', 'share of degenerate cells')]:
    for ycol in ['ece_slope', 'ece_delta']:
        r_s = spearmanr(S[xcol], S[ycol])
        r_p = pearsonr(S[xcol], S[ycol])
        print(f'  {lab_:<26} vs {ycol:<10}  '
              f'Spearman {r_s.correlation:+.3f} (p={r_s.pvalue:.2e})   '
              f'Pearson {r_p[0]:+.3f}')
    print()

print('=== within each design separately ===')
print('(this is the part that is free of the dataset confound)\n')
for d in S['design'].unique():
    sub = S[S.design == d]
    print(f'--- {d}, n = {len(sub)} ---')
    for xcol, lab_ in [('auroc_mean','transfer AUROC'),
                       ('degenerate_share','degeneracy')]:
        r = spearmanr(sub[xcol], sub['ece_slope'])
        print(f'   {lab_:<18} vs ece_slope  '
              f'Spearman {r.correlation:+.3f}  p={r.pvalue:.2e}')
    print()

print('=== the decisive comparison ===')
print('Within Camelyon17 alone the dataset, the task and the kind of shift are')
print('all constant. Any relationship there is transfer quality, not design.\n')
cv = S[S.design == 'covariate']
q = pd.qcut(cv['auroc_mean'], 4, labels=['worst','low','high','best'])
print(cv.groupby(q).agg(
    n=('rises','size'), rises=('rises','sum'),
    mean_auroc=('auroc_mean','mean'),
    mean_delta=('ece_delta','mean'),
    mean_slope=('ece_slope','mean')).round(4).to_string())

=== across all 192 combinations ===

  transfer AUROC             vs ece_slope   Spearman -0.782 (p=8.42e-41)   Pearson -0.806
  transfer AUROC             vs ece_delta   Spearman -0.798 (p=1.40e-43)   Pearson -0.818

  share of degenerate cells  vs ece_slope   Spearman +0.572 (p=4.57e-18)   Pearson +0.662
  share of degenerate cells  vs ece_delta   Spearman +0.576 (p=2.38e-18)   Pearson +0.664

=== within each design separately ===
(this is the part that is free of the dataset confound)

--- label-definition, n = 32 ---
   transfer AUROC     vs ece_slope  Spearman -0.053  p=7.73e-01
   degeneracy         vs ece_slope  Spearman +0.279  p=1.23e-01

--- covariate, n = 160 ---
   transfer AUROC     vs ece_slope  Spearman -0.647  p=2.42e-20
   degeneracy         vs ece_slope  Spearman +0.235  p=2.78e-03

=== the decisive comparison ===
Within Camelyon17 alone the dataset, the task and the kind of shift are
all constant. Any relationship there is transfer quality, not design.

             

## 3. Per model, within Camelyon17

Eight models on the same twenty pairs. If the effect belongs to transfer
quality, the models that transfer worst should show it most.

In [4]:
pm = cv.groupby('model').agg(
    pairs=('rises','size'), rises=('rises','sum'),
    ood_auroc=('auroc_mean','mean'),
    mean_delta=('ece_delta','mean'),
    mean_slope=('ece_slope','mean')).sort_values('ood_auroc')
print(pm.round(4).to_string())

r = spearmanr(pm['ood_auroc'], pm['rises'])
print(f'\n  across the 8 models: OOD AUROC vs number of rising pairs, '
      f'Spearman {r.correlation:+.3f}  p={r.pvalue:.3f}')
r2 = spearmanr(pm['ood_auroc'], pm['mean_slope'])
print(f'  OOD AUROC vs mean ECE slope, '
      f'Spearman {r2.correlation:+.3f}  p={r2.pvalue:.3f}')

print('\n=== the same, per pair rather than per model ===')
pp = cv.groupby('pair').agg(
    models=('rises','size'), rises=('rises','sum'),
    ood_auroc=('auroc_mean','mean'),
    mean_delta=('ece_delta','mean')).sort_values('ood_auroc')
print(pp.round(4).to_string())
r3 = spearmanr(pp['ood_auroc'], pp['rises'])
print(f'\n  across the 20 pairs: Spearman {r3.correlation:+.3f}  p={r3.pvalue:.3f}')

             pairs  rises  ood_auroc  mean_delta  mean_slope
model                                                       
CLIP            20      8     0.8490      0.0008      0.0008
PLIP            20      1     0.9660     -0.1635     -0.0818
CONCH           20      0     0.9836     -0.1450     -0.0714
MIDNIGHT        20      0     0.9892     -0.1517     -0.0750
GigaPath        20      1     0.9916     -0.1696     -0.0846
H-Optimus-0     20      0     0.9923     -0.1762     -0.0885
UNI             20      0     0.9938     -0.1966     -0.0986
VIRCHOW2        20      0     0.9946     -0.1921     -0.0958

  across the 8 models: OOD AUROC vs number of rising pairs, Spearman -0.674  p=0.067
  OOD AUROC vs mean ECE slope, Spearman -0.905  p=0.002

=== the same, per pair rather than per model ===
        models  rises  ood_auroc  mean_delta
pair                                        
h4->h1       8      1     0.9293     -0.1006
h1->h4       8      2     0.9308     -0.0743
h2->h1       8    

## 4. Is there a transfer-quality threshold?

If the effect switches sign at a particular level of transfer performance, that
is a statement a practitioner can act on.

In [5]:
print('=== rate of rising ECE by transfer AUROC band, both designs ===\n')
bands = [(0.0,0.55),(0.55,0.70),(0.70,0.85),(0.85,0.95),(0.95,1.01)]
rows = []
for lo, hi in bands:
    sub = S[(S.auroc_mean >= lo) & (S.auroc_mean < hi)]
    if len(sub) == 0:
        continue
    rows.append(dict(band=f'[{lo:.2f}, {hi:.2f})', n=len(sub),
                     rises=int(sub.rises.sum()),
                     rate=round(100*sub.rises.mean(), 1),
                     mean_delta=round(sub.ece_delta.mean(), 4),
                     designs=', '.join(sorted(sub.design.unique()))))
band_tab = pd.DataFrame(rows)
print(band_tab.to_string(index=False))
band_tab.to_csv(f'{OUT_DIR}/mechanism_bands.csv', index=False)

print('\n=== where does the sign flip? ===')
srt = S.sort_values('auroc_mean')
for thr in [0.60, 0.70, 0.80, 0.90, 0.95]:
    below = srt[srt.auroc_mean < thr]
    above = srt[srt.auroc_mean >= thr]
    if len(below) < 5 or len(above) < 5:
        continue
    print(f'  AUROC < {thr:.2f}: {below.rises.mean()*100:>5.1f}% rise '
          f'(n={len(below)}, mean delta {below.ece_delta.mean():+.4f})   |   '
          f'>= {thr:.2f}: {above.rises.mean()*100:>5.1f}% rise '
          f'(n={len(above)}, mean delta {above.ece_delta.mean():+.4f})')

print('\n=== do rising and falling combinations differ in transfer quality? ===')
u = mannwhitneyu(S[S.rises].auroc_mean, S[~S.rises].auroc_mean)
print(f'  rising    n={S.rises.sum():>3}  '
      f'median AUROC {S[S.rises].auroc_mean.median():.3f}')
print(f'  falling   n={(~S.rises).sum():>3}  '
      f'median AUROC {S[~S.rises].auroc_mean.median():.3f}')
print(f'  Mann-Whitney U={u.statistic:.0f}, p={u.pvalue:.2e}')

=== rate of rising ECE by transfer AUROC band, both designs ===

        band   n  rises  rate  mean_delta                     designs
[0.00, 0.55)  16     14  87.5      0.0808            label-definition
[0.55, 0.70)  17     15  88.2      0.0892 covariate, label-definition
[0.70, 0.85)   7      6  85.7      0.1098 covariate, label-definition
[0.85, 0.95)  13      1   7.7     -0.0735                   covariate
[0.95, 1.01) 139      2   1.4     -0.1718                   covariate

=== where does the sign flip? ===
  AUROC < 0.60:  87.0% rise (n=23, mean delta +0.0786)   |   >= 0.60:  10.7% rise (n=169, mean delta -0.1365)
  AUROC < 0.70:  87.9% rise (n=33, mean delta +0.0852)   |   >= 0.70:   5.7% rise (n=159, mean delta -0.1514)
  AUROC < 0.80:  87.2% rise (n=39, mean delta +0.0916)   |   >= 0.80:   2.6% rise (n=153, mean delta -0.1623)
  AUROC < 0.90:  81.4% rise (n=43, mean delta +0.0815)   |   >= 0.90:   2.0% rise (n=149, mean delta -0.1662)
  AUROC < 0.95:  67.9% rise (n=53, mean 

## 5. Degeneracy: cause, correlate, or neither?

Single-class collapse occurs in 57% of label-definition cells and 0.7% of
hospital cells, so it is a candidate explanation. The question is whether the
effect survives when collapse is excluded.

In [6]:
print('=== restrict to combinations with no collapse at all ===\n')
clean = S[S.degenerate_share == 0]
print(f'  {len(clean)} of {len(S)} combinations never collapse')
print(clean.groupby('design').agg(
    n=('rises','size'), rises=('rises','sum'),
    mean_delta=('ece_delta','mean'),
    mean_auroc=('auroc_mean','mean')).round(4).to_string())

if len(clean[clean.design == 'label-definition']) >= 3:
    r = spearmanr(clean['auroc_mean'], clean['ece_slope'])
    print(f'\n  among non-collapsing combinations, AUROC vs slope: '
          f'Spearman {r.correlation:+.3f}  p={r.pvalue:.2e}')
else:
    print('\n  too few non-collapsing label-definition combinations to compare '
          'designs;\n  the covariate design carries this analysis')

print('\n=== partial picture: does AUROC still matter at fixed degeneracy? ===')
for lo, hi, name in [(0.0, 0.01, 'no collapse'),
                     (0.01, 0.50, 'some collapse'),
                     (0.50, 1.01, 'mostly collapsed')]:
    sub = S[(S.degenerate_share >= lo) & (S.degenerate_share < hi)]
    if len(sub) < 8:
        print(f'  {name:<18} n={len(sub)}, too few')
        continue
    r = spearmanr(sub['auroc_mean'], sub['ece_slope'])
    print(f'  {name:<18} n={len(sub):>3}  rises {sub.rises.mean()*100:>5.1f}%  '
          f'AUROC vs slope Spearman {r.correlation:+.3f} (p={r.pvalue:.3f})')

print('\n=== which explains more, transfer quality or degeneracy? ===')
try:
    import statsmodels.api as sm
    X = sm.add_constant(S[['auroc_mean', 'degenerate_share']])
    fit = sm.OLS(S['ece_slope'], X).fit()
    print(fit.summary().tables[1])
    print(f'\n  R-squared {fit.rsquared:.3f}')
    for c in ['auroc_mean', 'degenerate_share']:
        Xs = sm.add_constant(S[[c]])
        print(f'  {c:<20} alone: R-squared '
              f'{sm.OLS(S["ece_slope"], Xs).fit().rsquared:.3f}')
except ImportError:
    print('  statsmodels not available; install with !pip install statsmodels')

=== restrict to combinations with no collapse at all ===

  163 of 192 combinations never collapse
                    n  rises  mean_delta  mean_auroc
design                                              
covariate         157      7     -0.1565      0.9757
label-definition    6      5      0.0563      0.5121

  among non-collapsing combinations, AUROC vs slope: Spearman -0.664  p=4.21e-22

=== partial picture: does AUROC still matter at fixed degeneracy? ===
  no collapse        n=163  rises   7.4%  AUROC vs slope Spearman -0.664 (p=0.000)
  some collapse      n= 12  rises  83.3%  AUROC vs slope Spearman +0.231 (p=0.471)
  mostly collapsed   n= 17  rises  94.1%  AUROC vs slope Spearman +0.108 (p=0.680)

=== which explains more, transfer quality or degeneracy? ===
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.1522      0.019      7.858      

## 6. The numbers the paper will quote

Everything a rewritten Section 4 needs, in one place.

In [7]:
print('=' * 68)
print('CONTRAST')
print('=' * 68)
for d in ['label-definition', 'covariate']:
    s = S[S.design == d]
    print(f'  {d:<18} {s.rises.sum():>3}/{len(s):<4} rise '
          f'({100*s.rises.mean():>5.1f}%)   '
          f'mean delta {s.ece_delta.mean():+.4f}   '
          f'mean transfer AUROC {s.auroc_mean.mean():.3f}')

CLAIM = ['UNI','VIRCHOW2','GigaPath','H-Optimus-0']
print('\n  four claim models only:')
for d in ['label-definition', 'covariate']:
    s = S[(S.design == d) & (S.model.isin(CLAIM))]
    print(f'    {d:<18} {s.rises.sum():>3}/{len(s):<4}   '
          f'mean delta {s.ece_delta.mean():+.4f}')

print('\n' + '=' * 68)
print('MECHANISM')
print('=' * 68)
r_all = spearmanr(S['auroc_mean'], S['ece_slope'])
r_cov = spearmanr(cv['auroc_mean'], cv['ece_slope'])
print(f'  transfer AUROC vs ECE slope, all 192      '
      f'Spearman {r_all.correlation:+.3f}  p={r_all.pvalue:.2e}')
print(f'  same, within Camelyon17 alone (n={len(cv)})    '
      f'Spearman {r_cov.correlation:+.3f}  p={r_cov.pvalue:.2e}')
print(f'  degeneracy: {100*S[S.design=="label-definition"].degenerate_share.mean():.1f}% '
      f'of label-definition cells, '
      f'{100*S[S.design=="covariate"].degenerate_share.mean():.1f}% of hospital cells')

print('\n' + '=' * 68)
print('THE CLIP CASE')
print('=' * 68)
c = cv[cv.model == 'CLIP']
o = cv[cv.model != 'CLIP']
print(f'  CLIP        OOD AUROC {c.auroc_mean.mean():.3f}   '
      f'rises {c.rises.sum()}/{len(c)}   mean delta {c.ece_delta.mean():+.4f}')
print(f'  other 7     OOD AUROC {o.auroc_mean.mean():.3f}   '
      f'rises {o.rises.sum()}/{len(o)}   mean delta {o.ece_delta.mean():+.4f}')
print('\n  Same dataset, same pairs, same kind of shift. The only thing that')
print('  differs is how well the model transfers.')

S.to_csv(f'{OUT_DIR}/mechanism_summary.csv', index=False)
print('\nsaved -> mechanism_summary.csv, mechanism_bands.csv')

CONTRAST
  label-definition    28/32   rise ( 87.5%)   mean delta +0.0820   mean transfer AUROC 0.538
  covariate           10/160  rise (  6.2%)   mean delta -0.1492   mean transfer AUROC 0.970

  four claim models only:
    label-definition    14/16     mean delta +0.0909
    covariate            1/80     mean delta -0.1836

MECHANISM
  transfer AUROC vs ECE slope, all 192      Spearman -0.782  p=8.42e-41
  same, within Camelyon17 alone (n=160)    Spearman -0.647  p=2.42e-20
  degeneracy: 57.3% of label-definition cells, 0.7% of hospital cells

THE CLIP CASE
  CLIP        OOD AUROC 0.849   rises 8/20   mean delta +0.0008
  other 7     OOD AUROC 0.987   rises 2/140   mean delta -0.1707

  Same dataset, same pairs, same kind of shift. The only thing that
  differs is how well the model transfers.

saved -> mechanism_summary.csv, mechanism_bands.csv


## 7. How to read this

**If transfer AUROC predicts the ECE slope both overall and within Camelyon17
alone**, the paper's central claim becomes: *reverse calibration scaling is a
signature of failing transfer, and label-definition shift is the most reliable
way to produce it.* That statement survives the confound, is supported by 192
combinations, and gives a practitioner something to act on: if you observe
calibration worsening as you add source labels, your probe is not transferring,
whatever the shift is called.

**If the relationship holds across designs but not within Camelyon17**, the
dataset confound is not resolved and the paper should claim the contrast
without claiming the mechanism.

**If degeneracy explains more than transfer quality does**, say so plainly. The
finding is then that the effect is largely an artefact of single-class collapse
under severe shift, which is worth reporting and is what Reviewer tp5b
suspected.

The regression in section 5 answers the last question directly by comparing how
much of the ECE slope each variable accounts for.